# AUTO-RUN analysis
# ================
This notebook contains the POST-HOC ANALYSIS cells that should be run
after the pipeline completes. They read results.json and produce the
figures and tables for the thesis.


## POST-HOC ANALYSIS NOTEBOOK


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# ── UPDATE THIS PATH to your actual results.json ─────────────────────────────
RESULTS_PATH  = Path("/home/mjohn/thesis_models/auto-run/outputs/results.json")
RANKING_PATH  = Path("/home/mjohn/thesis_models/auto-run/outputs/rankings.json")
OUT_DIR       = Path("/home/mjohn/thesis_models/auto-run/outputs/analysis")
# ─────────────────────────────────────────────────────────────────────────────

OUT_DIR.mkdir(parents=True, exist_ok=True)

results  = json.loads(RESULTS_PATH.read_text())
rankings = json.loads(RANKING_PATH.read_text())

thresh_results = results["threshold_results"]
per_painting   = pd.DataFrame(results["per_painting"])

print(f"Loaded results: {len(per_painting)} scorable paintings")
print(f"Thresholds available: {list(thresh_results.keys())}")


## CELL 1 — IoU Threshold Sweep Table
### Main results table for Section 5.3

In [ ]:
# ── TABLE: IoU threshold sweep ────────────────────────────────────────────────
rows = []
for thresh_str, agg in thresh_results.items():
    thresh = float(thresh_str)
    n      = agg["n_scorable_paintings"]
    if n == 0:
        continue
    rows.append({
        "IoU threshold":              f"≥{thresh:.2f}",
        "Scorable paintings":         n,
        "Paintings with ≥1 match":    agg["n_paintings_with_match"],
        "P@1":   round(agg.get("mean_p_at_1", 0), 3),
        "P@3":   round(agg.get("mean_p_at_3", 0), 3),
        "R@1":   round(agg.get("mean_r_at_1", 0), 3),
        "R@3":   round(agg.get("mean_r_at_3", 0), 3),
        "MRR":   round(agg.get("mean_mrr", 0),   3),
        "Hit@3": round(agg.get("mean_hit_at_3", 0), 3),
    })

sweep_df = pd.DataFrame(rows)
print("=" * 80)
print("AUTO-RUN: IoU threshold sweep (pure IoU matching)")
print("=" * 80)
display(sweep_df)
sweep_df.to_csv(OUT_DIR / "iou_sweep_table.csv", index=False)
print(f"Saved: {OUT_DIR / 'iou_sweep_table.csv'}")

## CELL 2 — IoU Sweep Plots
Two-panel figure: metrics vs threshold + coverage vs threshold

In [ ]:
# ── PLOT: metric drop + coverage drop across thresholds ──────────────────────
thresholds    = [float(t) for t in thresh_results]
thresh_labels = [f"≥{t:.2f}" for t in thresholds]

def get_metric(thresh, key):
    agg = thresh_results[str(thresh)]
    return agg.get(f"mean_{key}", 0.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left panel: metrics vs threshold
ax = axes[0]
metric_specs = [
    ("p_at_1",   "P@1",   "#2166ac", "o"),
    ("mrr",      "MRR",   "#4dac26", "s"),
    ("hit_at_3", "Hit@3", "#d01c8b", "^"),
    ("r_at_3",   "R@3",   "#e67e22", "D"),
]
x = np.arange(len(thresholds))
for key, label, color, marker in metric_specs:
    vals = [get_metric(t, key) for t in thresholds]
    ax.plot(x, vals, marker=marker, color=color, label=label,
            linewidth=2, markersize=8)

ax.set_xticks(x)
ax.set_xticklabels(thresh_labels, fontsize=10)
ax.set_xlabel("IoU threshold", fontsize=10)
ax.set_ylabel("Score", fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_title("Auto-run metrics vs IoU threshold", fontsize=11)
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)

# Annotate values
for key, label, color, marker in metric_specs:
    vals = [get_metric(t, key) for t in thresholds]
    for xi, v in zip(x, vals):
        ax.annotate(f"{v:.2f}", (xi, v), textcoords="offset points",
                    xytext=(0, 7), ha="center", fontsize=7, color=color)

# Right panel: matched paintings count vs threshold
ax2 = axes[1]
match_counts    = [thresh_results[str(t)]["n_paintings_with_match"] for t in thresholds]
scorable_counts = [thresh_results[str(t)]["n_scorable_paintings"]   for t in thresholds]

bars = ax2.bar(x, match_counts, color="#2166ac", alpha=0.75,
               edgecolor="#333", label="Paintings with ≥1 match")
ax2.bar(x, scorable_counts, color="#aec7e8", alpha=0.4,
        edgecolor="#333", label="All scorable paintings")

for xi, v in zip(x, match_counts):
    ax2.text(xi, v + 3, str(v), ha="center", fontsize=9, fontweight="bold")

ax2.set_xticks(x)
ax2.set_xticklabels(thresh_labels, fontsize=10)
ax2.set_ylabel("Number of paintings", fontsize=10)
ax2.set_title("Match coverage vs IoU threshold", fontsize=11)
ax2.legend(fontsize=9)
ax2.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    "Auto-run IoU threshold sweep\n"
    "(drop in coverage quantifies ArtDL annotation scale mismatch with YOLO)",
    fontsize=11, y=1.01
)
plt.tight_layout()
sweep_plot = OUT_DIR / "iou_sweep_plot.png"
plt.savefig(sweep_plot, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {sweep_plot}")

## CELL 3 — R@1 Ceiling Analysis
Shows R@1 ≈ theoretical ceiling (1 / mean saints per painting)
Use in thesis to explain why R@1 is low despite high P@1

In [ ]:
# ── R@1 CEILING ANALYSIS ─────────────────────────────────────────────────────
# Explains the P@1 high / R@1 low apparent paradox.

n_artdl_per_painting = per_painting["n_artdl_boxes"].values
mean_saints = n_artdl_per_painting.mean()
theoretical_r1_ceiling = 1.0 / mean_saints
theoretical_r3_ceiling = min(3.0 / mean_saints, 1.0)

# Primary threshold (0.20)
primary_thresh = str(min(float(t) for t in thresh_results))
observed_r1 = thresh_results[primary_thresh].get("mean_r_at_1", 0)
observed_r3 = thresh_results[primary_thresh].get("mean_r_at_3", 0)
observed_p1 = thresh_results[primary_thresh].get("mean_p_at_1", 0)

print("=" * 60)
print("R@K CEILING ANALYSIS")
print("=" * 60)
print(f"Mean saints per painting (ArtDL):  {mean_saints:.2f}")
print(f"Theoretical R@1 ceiling:           1/{mean_saints:.1f} = {theoretical_r1_ceiling:.3f}")
print(f"Theoretical R@3 ceiling:           3/{mean_saints:.1f} = {theoretical_r3_ceiling:.3f}")
print()
print(f"Observed P@1  (IoU ≥ 0.20):        {observed_p1:.3f}")
print(f"Observed R@1  (IoU ≥ 0.20):        {observed_r1:.3f}  ({observed_r1/theoretical_r1_ceiling*100:.0f}% of ceiling)")
print(f"Observed R@3  (IoU ≥ 0.20):        {observed_r3:.3f}  ({observed_r3/theoretical_r3_ceiling*100:.0f}% of ceiling)")
print()
print("Interpretation:")
print(f"  R@1 ≈ {observed_r1:.3f} is essentially the hard ceiling ({theoretical_r1_ceiling:.3f}).")
print(f"  The pipeline reliably finds ONE saint but was not designed to recover ALL {mean_saints:.1f}.")
print(f"  High P@1 ({observed_p1:.3f}) and low R@1 ({observed_r1:.3f}) are structurally expected,")
print( "  not a contradiction.")

# ── Histogram: saints per painting ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
unique, counts = np.unique(n_artdl_per_painting, return_counts=True)
ax.bar(unique, counts, color="#2166ac", alpha=0.8, edgecolor="#333")
ax.axvline(mean_saints, color="#d01c8b", linewidth=2,
           linestyle="--", label=f"Mean = {mean_saints:.1f}")
ax.set_xlabel("Number of annotated saints per painting", fontsize=10)
ax.set_ylabel("Number of paintings", fontsize=10)
ax.set_title("ArtDL annotation density\n"
             "(explains low R@K despite high P@1)", fontsize=11)
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
density_plot = OUT_DIR / "artdl_annotation_density.png"
plt.savefig(density_plot, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved: {density_plot}")

In [ ]:
# ── SAINTS-TO-DETECTIONS RATIO ──────────────────────────────────────────────
rankings = json.load(open(RANKING_PATH))
dets = [len(v['ranked_detections']) for v in rankings.values() if v['status'] == 'ranked']

mean_det = np.mean(dets)
mean_saints = 4.74

print(f"Paintings with detections:    {len(dets)}")
print(f"Mean detections per painting: {mean_det:.1f}")
print(f"Mean saints per painting:     {mean_saints:.1f}")
print(f"Saints / detections ratio:    {mean_saints/mean_det:.2f}")
print(f"\nMore saints than detections: YOLO misses most saints entirely.")
print(f"Low recall reflects the detection gap, not poor ranking.")

---
## CELL 4 — Per-painting P@1 distribution
### Shows how P@1 is distributed across paintings at primary threshold

In [ ]:
# ── PER-PAINTING P@1 DISTRIBUTION ────────────────────────────────────────────
# Most paintings either score 1.0 (saint found at rank 1) or 0.0 (not found).
# This bar shows what fraction fall into each bucket at the primary threshold.

primary_col = "thresh_0.2_p_at_1"   # adjust if your primary threshold differs
if primary_col not in per_painting.columns:
    # Try to find the right column name
    p1_cols = [c for c in per_painting.columns if "p_at_1" in c]
    print(f"Available P@1 columns: {p1_cols}")
    primary_col = p1_cols[0] if p1_cols else None

if primary_col:
    vals = per_painting[primary_col].dropna()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(vals, bins=20, color="#2166ac", alpha=0.8, edgecolor="#333")
    ax.axvline(vals.mean(), color="#d01c8b", linewidth=2,
               linestyle="--", label=f"Mean P@1 = {vals.mean():.3f}")
    ax.set_xlabel("P@1 per painting", fontsize=10)
    ax.set_ylabel("Number of paintings", fontsize=10)
    ax.set_title(f"Per-painting P@1 distribution\n(IoU ≥ 0.20, n={len(vals)} paintings)",
                 fontsize=11)
    ax.legend(fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    p1_dist_plot = OUT_DIR / "autorun_p1_distribution.png"
    plt.savefig(p1_dist_plot, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {p1_dist_plot}")

## CELL 4 — Random Baseline

In [ ]:
# Expected P@1 under random ranking
# = probability that the single top-ranked figure is a matched saint
# = n_matched_saints / n_detections
# But since we don't know n_matched a priori, use Monte Carlo like the full run

def random_baseline_autorun(rankings, artdl_boxes_dict, 
                             iou_thresh, top_k_values,
                             n_sims=1000, seed=42):
    """
    Monte Carlo random baseline for the auto-run.
    For each painting, randomly permutes the detected figures 1000 times
    and averages the metrics. Uses the same IoU matching as evaluate.py
    but with shuffled ranks instead of composite-score ranks.
    """
    rng = np.random.default_rng(seed)
    metric_sums = defaultdict(float)
    n_scorable = 0

    for image_id, entry in rankings.items():
        ranked = entry.get("ranked_detections", [])
        gt_boxes = artdl_boxes_dict.get(image_id, [])
        if not ranked or not gt_boxes:
            continue

        iou_pairs = compute_all_ious(ranked, gt_boxes)
        n_det = len(ranked)
        n_total = len(gt_boxes)
        n_scorable += 1

        sim_metrics = defaultdict(list)
        for _ in range(n_sims):
            # Shuffle detection ranks randomly
            shuffled_ranks = rng.permutation(n_det) + 1.0

            # Apply shuffled ranks to iou_pairs
            shuffled_pairs = [
                {**p, "det_rank": float(shuffled_ranks[p["det_idx"]])}
                for p in iou_pairs
            ]
            matched_ranks = match_at_threshold(
                shuffled_pairs, gt_boxes, ranked, iou_thresh
            )
            m = compute_metrics(matched_ranks, n_total, top_k_values)
            for key, val in m.items():
                sim_metrics[key].append(val)

        for key, vals in sim_metrics.items():
            metric_sums[key] += float(np.mean(vals))

    if n_scorable == 0:
        return {}

    return {
        "n_scorable_paintings": n_scorable,
        **{f"mean_{k}": metric_sums[k] / n_scorable 
           for k in metric_sums}
    }